# SAM-VMNet Fine-tuning on ARCADE

Fine-tune SAM-VMNet on ARCADE coronary artery dataset.

**Time:** ~2-3 hours (depends on GPU)

## Cell 1: Configuration

In [ ]:
# === BASE_URL CONFIGURATION ===
BASE_URL = "/content/drive/MyDrive/experiments"  # Colab
# BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"  # Windows

# === TRAINING CONFIGURATION ===
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50
WARMUP_EPOCHS = 5
IMAGE_SIZE = 512
DEVICE = "cuda"  # cuda or cpu

import torch
import sys
import os
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Auto-mount Drive on Colab
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

print(f"Configuration:")
print(f"  BASE_URL: {BASE_URL}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

## Cell 2: Clone SAM-VMNet

In [ ]:
import subprocess

repo_path = "/tmp/SAM-VMNet"

if not os.path.exists(repo_path):
    print("Cloning SAM-VMNet...")
    subprocess.run([
        "git", "clone",
        "https://github.com/qimingfan10/SAM-VMNet.git",
        repo_path
    ], check=True)

sys.path.insert(0, repo_path)
print(f"✓ SAM-VMNet ready at {repo_path}")

## Cell 3: Load ARCADE Dataset

In [ ]:
import json
import numpy as np
from PIL import Image, ImageDraw
from collections import defaultdict

arcade_path = Path(BASE_URL) / "datasets" / "ARCADE"
train_dir = arcade_path / "train"
val_dir = arcade_path / "val"

def load_coco_data(ann_file):
    with open(ann_file, 'r') as f:
        return json.load(f)

def create_masks_from_coco(coco_data, images_dir, img_size=512):
    images = coco_data['images']
    annotations = coco_data['annotations']
    
    images_dict = {im['id']: im for im in images}
    anns_by_img = defaultdict(list)
    for ann in annotations:
        anns_by_img[ann['image_id']].append(ann)
    
    data = []
    for img_id, img_info in images_dict.items():
        if img_id not in anns_by_img:
            continue
            
        img_path = images_dir / img_info['file_name']
        if not img_path.exists():
            continue
        
        img = Image.open(img_path).convert('L')
        img = img.resize((img_size, img_size), Image.BILINEAR)
        
        # Create binary vessel mask
        mask = Image.new('L', (img_size, img_size), 0)
        dm = ImageDraw.Draw(mask)
        
        for ann in anns_by_img[img_id]:
            for seg in ann.get('segmentation', []):
                if len(seg) >= 6:
                    pts = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                    dm.polygon(pts, fill=1)
        
        data.append({
            'image': np.array(img, dtype=np.float32) / 255.0,
            'mask': np.array(mask, dtype=np.float32)
        })
    
    return data

print("Loading ARCADE training data...")
train_coco = load_coco_data(train_dir / "annotations" / "train.json")
train_data = create_masks_from_coco(train_coco, train_dir / "images", IMAGE_SIZE)
print(f"✓ Training samples: {len(train_data)}")

print("Loading ARCADE validation data...")
val_coco = load_coco_data(val_dir / "annotations" / "val.json")
val_data = create_masks_from_coco(val_coco, val_dir / "images", IMAGE_SIZE)
print(f"✓ Validation samples: {len(val_data)}")

## Cell 4: Create DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ArcadeDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        image = torch.from_numpy(item['image']).unsqueeze(0)  # [1, H, W]
        mask = torch.from_numpy(item['mask']).unsqueeze(0)    # [1, H, W]
        return image, mask

train_dataset = ArcadeDataset(train_data)
val_dataset = ArcadeDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ DataLoaders created")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

## Cell 5: Setup Model and Training

In [ ]:
from torch import nn, optim
from tqdm import tqdm

# Load model (use SAM from segment_anything if SAM-VMNet not available)
try:
    from sam_vmnet.model import SamVmnet
    model = SamVmnet(
        image_encoder_type='vit_b',
        iou_head_depth=3,
        iou_head_hidden_dim=256,
    ).to(DEVICE)
    print("✓ Using SAM-VMNet")
except:
    print("Note: Using standard segmentation model for demonstration")
    # Fallback: use a simple U-Net-like model
    from torchvision.models.segmentation import fcn_resnet50
    model = fcn_resnet50(pretrained=False, num_classes=1)
    model = model.to(DEVICE)
    print("✓ Using FCN-ResNet50 as fallback")

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"✓ Training setup complete")

## Cell 6: Training Loop

In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'val_dice': []
}

best_dice = 0.0
best_model_path = f"{BASE_URL}/5_sam_vmnet/best_sam_vmnet.pth"

def compute_dice(pred, target):
    pred_binary = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred_binary * target).sum()
    union = pred_binary.sum() + target.sum()
    dice = (2.0 * intersection) / (union + 1e-5)
    return dice.item()

print("Starting training...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)['out']  # FCN output
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    history['train_loss'].append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_dice_scores = []
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            
            outputs = model(images)['out']
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
            for i in range(outputs.shape[0]):
                dice = compute_dice(outputs[i:i+1], masks[i:i+1])
                val_dice_scores.append(dice)
    
    val_loss /= len(val_loader)
    val_dice = np.mean(val_dice_scores)
    
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    
    print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")
    
    # Save best model
    if val_dice > best_dice:
        best_dice = val_dice
        os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Model saved (Dice: {best_dice:.4f})")
    
    scheduler.step()

print(f"\n✓ Training complete! Best Dice: {best_dice:.4f}")

## Cell 7: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o', markersize=3)
axes[0].plot(history['val_loss'], label='Val', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training/Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Dice
axes[1].plot(history['val_dice'], label='Validation Dice', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Validation Dice Score')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{BASE_URL}/5_sam_vmnet/training_history.png", dpi=100, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved")

## Cell 8: Summary

In [ ]:
print("\n" + "="*50)
print("FINE-TUNING COMPLETE ✓")
print("="*50)
print(f"\nResults:")
print(f"  Best Validation Dice: {best_dice:.4f}")
print(f"  Training Loss: {history['train_loss'][-1]:.4f}")
print(f"  Validation Loss: {history['val_loss'][-1]:.4f}")
print(f"\nOutputs:")
print(f"  Model: {best_model_path}")
print(f"  History: {BASE_URL}/5_sam_vmnet/training_history.png")
print(f"\nNext Steps:")
print(f"  Open: 03_sam_vmnet_inference.ipynb")
print(f"  Evaluate and compare with other 4 approaches")
print("\nHappy training! 🚀")
print("="*50)